# Huấn luyện mô hình YOLOv8 (Object Detection)
Sử dụng cho nhận diện rác nổi, tàn thuốc, vật thể dị thường. Notebook này được tối ưu sẵn cho card RTX 3050 (batch_size nhỏ, mixed precision).

In [6]:
# 1. Cài đặt thư viện YOLOv8 và các package tải dataset, xử lý file (nếu chưa cài)
# %pip install ultralytics roboflow kagglehub pyyaml

In [7]:
from ultralytics import YOLO

# 2. Khởi tạo mô hình
# Sử dụng phiên bản Nano (yolov8n.pt) cực nhẹ, phù hợp cho bài toán real-time trên phần cứng giới hạn
model = YOLO('yolov8s.pt')

In [ ]:
# 3. Gộp nhãn và Dữ liệu từ Roboflow (Clean Floor) & Kaggle (Trash Detection)
import os
import yaml
import shutil
from pathlib import Path

workspace_root = Path("..").resolve()
yolo_dir = str((workspace_root / "data" / "raw" / "yolo").resolve())
roboflow_dir = os.path.join(yolo_dir, "roboflow")
kaggle_dir = os.path.join(yolo_dir, "kaggle", "Dataset")
merged_dir = os.path.join(yolo_dir, "merged")

print("⏳ Đang dọn dẹp thư mục gộp cũ (nếu có)...")
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir)
os.makedirs(merged_dir, exist_ok=True)

# Tạo cấu trúc thư mục YOLO chuẩn
for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(merged_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(merged_dir, "labels", split), exist_ok=True)

# =============== LOGIC MAP NHÃN =================
# Roboflow: 0:metal, 1:paper, 2:plastic, 3:trash
# Kaggle: 0:dirt, 1:liquid, 2:marks, 3:trash
# Nhãn thông minh sau khi gộp (Trash số 3 trùng nhau nên giữ nguyên):
# 0:metal, 1:paper, 2:plastic, 3:trash, 4:dirt, 5:liquid, 6:marks
unified_classes = ["metal", "paper", "plastic", "trash", "dirt", "liquid", "marks"]
map_rf = {0: 0, 1: 1, 2: 2, 3: 3}
map_kg = {0: 4, 1: 5, 2: 6, 3: 3}

def copy_and_map(src_img_dir, src_lbl_dir, dest_split, map_dict, prefix=""):
    if not os.path.exists(src_img_dir) or not os.path.exists(src_lbl_dir):
        return
    dest_img_dir = os.path.join(merged_dir, "images", dest_split)
    dest_lbl_dir = os.path.join(merged_dir, "labels", dest_split)

    count = 0
    for filename in os.listdir(src_img_dir):
        if not filename.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        # 1. Copy và đổi tên ảnh tránh trùng lặp
        new_filename = f"{prefix}{filename}"
        shutil.copy(os.path.join(src_img_dir, filename), os.path.join(dest_img_dir, new_filename))

        # 2. Xử lý file Nhãn (Txt)
        lbl_name = os.path.splitext(filename)[0] + ".txt"
        src_lbl_path = os.path.join(src_lbl_dir, lbl_name)
        dest_lbl_path = os.path.join(dest_lbl_dir, f"{prefix}{lbl_name}")

        if os.path.exists(src_lbl_path):
            with open(src_lbl_path, "r") as fin:
                lines = fin.readlines()
            with open(dest_lbl_path, "w") as fout:
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        old_id = int(parts[0])
                        if old_id in map_dict:
                            fout.write(f"{map_dict[old_id]} " + " ".join(parts[1:]) + "\n")
            count += 1
    return count

print("⏳ Đang xử lý data từ Roboflow (Clean Floor)...")
c1 = copy_and_map(os.path.join(roboflow_dir, "train", "images"), os.path.join(roboflow_dir, "train", "labels"), "train", map_rf, "rf_")
c2 = copy_and_map(os.path.join(roboflow_dir, "valid", "images"), os.path.join(roboflow_dir, "valid", "labels"), "valid", map_rf, "rf_")
c3 = copy_and_map(os.path.join(roboflow_dir, "test", "images"), os.path.join(roboflow_dir, "test", "labels"), "test", map_rf, "rf_")

print("⏳ Đang xử lý data từ Kaggle (Trash Detection)...")
c4 = copy_and_map(os.path.join(kaggle_dir, "images", "train"), os.path.join(kaggle_dir, "labels", "train"), "train", map_kg, "kg_")
c5 = copy_and_map(os.path.join(kaggle_dir, "images", "val"), os.path.join(kaggle_dir, "labels", "val"), "valid", map_kg, "kg_")

# 3. Tạo file cấu hình data.yaml tổng hợp
yaml_content = {
    "path": merged_dir,
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(unified_classes)},
}
data_path = os.path.join(merged_dir, "data_merged.yaml")
with open(data_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\n✅ Đã trích xuất & Gộp {c1+c2+c3+c4+c5} ảnh/nhãn thành công!")
print(f"✅ Bộ dữ liệu Huấn luyện đã sẵn sàng ở: {data_path}")
print("🔹 Danh sách 7 Nhãn Tối Ưu (đã triệt tiêu lặp nhãn Trash):")
for k, v in yaml_content["names"].items():
    print(f"   [{k}]: {v}")

⏳ Đang dọn dẹp thư mục gộp cũ (nếu có)...
⏳ Đang xử lý data từ Roboflow (Clean Floor)...
⏳ Đang xử lý data từ Kaggle (Trash Detection)...

✅ Đã trích xuất & Gộp 3567 ảnh/nhãn thành công!
✅ Bộ dữ liệu Huấn luyện đã sẵn sàng ở: E:\capstone\cleaning_ai_poc\yolo_dataset\merged\data_merged.yaml
🔹 Danh sách 7 Nhãn Tối Ưu (đã triệt tiêu lặp nhãn Trash):
   [0]: metal
   [1]: paper
   [2]: plastic
   [3]: trash
   [4]: dirt
   [5]: liquid
   [6]: marks


In [ ]:
# 4. Bắt đầu quá trình Huấn luyện (Training)
import os
from pathlib import Path
import shutil

# Optional: append reviewed bridge negatives into merged dataset before training
bridge_root = Path("..") / "data" / "retrain_bridge" / "yolo"
if bridge_root.exists():
    print(f"🔁 Đang append reviewed bridge data từ: {bridge_root.resolve()}")
    split_map = {"train": "train", "valid": "valid", "test": "test"}
    for src_split, dst_split in split_map.items():
        src_img_dir = bridge_root / "images" / src_split
        src_lbl_dir = bridge_root / "labels" / src_split
        dst_img_dir = Path(merged_dir) / "images" / dst_split
        dst_lbl_dir = Path(merged_dir) / "labels" / dst_split

        if not src_img_dir.exists():
            continue

        for src_img in src_img_dir.glob("*"):
            if src_img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
                continue
            target_img = dst_img_dir / f"rv_{src_img.name}"
            shutil.copy2(src_img, target_img)

            src_label = src_lbl_dir / f"{src_img.stem}.txt"
            target_label = dst_lbl_dir / f"rv_{src_img.stem}.txt"
            if src_label.exists():
                shutil.copy2(src_label, target_label)
            else:
                target_label.write_text("", encoding="utf-8")
else:
    print("ℹ️ Không tìm thấy bridge dataset, bỏ qua bước append reviewed data.")

# Lấy đường dẫn tuyệt đối ép YOLO lưu đúng dự án hiện tại
project_dir = str((Path("..") / "outputs" / "yolo_training_4_4").resolve())

results = model.train(
    data=data_path,
    epochs=100,
    patience=30,  # Nếu 30 vòng liên tiếp không tiến bộ thì tự ngắt
    imgsz=640,
    batch=4,
    device=0,
    half=True,
    workers=0,  # FIX LỖI MEMORY ERROR: Tắt song song CPU trên Windows!
    project=project_dir,
    name="run_poc_1",
    verbose=True,
)

Ultralytics 8.4.33  Python-3.11.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\capstone\cleaning_ai_poc\yolo_dataset\merged\data_merged.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_poc_12, nbs=64, nms=False, opset=None, optimize=False, op

In [10]:
# 5. Kiểm thử nhanh kết quả trên 1 tấm ảnh (Inference)
test_image = '../data/raw/samples/test_after.jpg' # Thay bằng ảnh test thực tế của bạn

if os.path.exists(test_image): 
    res = model.predict(source=test_image, conf=0.25, save=True, project='../outputs/yolo_inference')
    print("Hoàn tất dự đoán! Kiểm tra thư mục outputs/yolo_inference để xem ảnh kết quả.")